# Cartilage Indentation Analysis — interactive notebook

This notebook is a **thin interface** to the `cartilage_indentation` package: it imports the
reusable code and runs the analysis step by step, showing the fitted parameters and figures
inline. All the actual logic lives in the package (`../cartilage_indentation/`).

## Before you run: how to set up your data

The pipeline reads two things from the `data/` folder — **you do not set any absolute paths**:

- **`data/raw/`** — your raw Bioindenter `.TXT` files, **one per sample**.
- **`data/sample_groups.xlsx`** — a table with **one row per file**, telling the pipeline what each file is.

### `sample_groups.xlsx` columns

| column | example | meaning |
| --- | --- | --- |
| `file_name` | `sample_01.TXT` | must match a file in `data/raw/` **exactly** |
| `sample_id` | `1` | unique integer; the key used for the results |
| `treatment` | `Group A Native` | free text describing the sample |
| `test_type` | `mono` **or** `stress_relaxation` | picks the analysis: Hertz (mono) or Prony (SR) |
| `Group` | `Group A Native Mono` | grouping label (usually `treatment` + `test_type`) |

Any extra columns you add (notes, dates, status, ...) are kept and carried through to the results.

### Native-vs-UV pairing (important)

For the comparison figures, the pipeline auto-pairs a `Native` group with its `UV` group by swapping
the word `Native` -> `UV`. The two group names must be **identical except for Native / UV**:

| Native group | UV group | pairs? |
| --- | --- | --- |
| `Group A Native Mono` | `Group A UV Mono` | yes |
| `Group A Native Mono` | `Group A UV  Mono` (double space) | yes (whitespace is auto-normalized) |
| `Group A Native Mono` | `Group A UV mono` (different word/case) | no |

**Every** mono sample gets a Hertz fit and **every** SR sample gets a Prony fit regardless of pairing;
pairing only controls the Native-vs-UV *comparison* figures. If a `Native` group has no matching `UV`
group, the run prints a warning naming it.

See **[`../data/README.md`](../data/README.md)** for the full spec (raw `.TXT` format, segments, units) and **`../config.py`** for the analysis parameters.

### Running on data in a separate folder

By default the pipeline uses the `data/` folder inside this repo. To analyse a **different** dataset without copying it into the repo, arrange it as `my_dataset/raw/` + `my_dataset/sample_groups.xlsx`, then either set `CIA_DATA_ROOT` in the setup cell below (one commented line), or in a terminal run `export CIA_DATA_ROOT=/path/to/my_dataset` before `python scripts/run_analysis.py`. Results are written to `my_dataset/Analysis_Output/` and the repo stays untouched. See `../config.py` for details.


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))   # make the repo root importable (config + package)

# --- OPTIONAL: run on data in a SEPARATE folder (not the repo's data/) ---
# Point at a folder that contains  raw/  and  sample_groups.xlsx .
# Outputs then go to <that folder>/Analysis_Output . Leave commented to use the repo data.
# Must be set BEFORE importing config.
# os.environ["CIA_DATA_ROOT"] = "/Users/you/path/to/my_new_dataset"

import config as cfg
from cartilage_indentation.dataset import SampleIndex, native_uv_pairs
from cartilage_indentation.preprocessing import preprocess_all
from cartilage_indentation.hertz import get_loading
from cartilage_indentation import plotting
from cartilage_indentation.results import save_results
from IPython.display import Image, display


def show_all(folder, pattern="*.png"):
    """Display every figure in a folder inline (in filename order)."""
    files = sorted(glob.glob(os.path.join(folder, pattern)))
    print(f"{len(files)} figure(s) in {os.path.basename(folder)}/")
    for f in files:
        print(os.path.basename(f))
        display(Image(f))

## 1. Load and group the data

In [ ]:
index = SampleIndex(cfg.SAMPLE_SHEET, cfg.RAW_DATA_DIR)
pairs = native_uv_pairs(index)
print(f"{len(index.files)} files, {len(index.group_order)} groups "
      f"({len(index.subgroup_mono)} mono, {len(index.subgroup_sr)} stress-relaxation), "
      f"{len(pairs)} Native/UV pairs")
index.sample_df.head()
# index.sample_df

## 2. Preprocess (drift correction + contact zeroing)

In [ ]:
mono_data, sr_data = preprocess_all(index, cfg)
print(f"preprocessed {len(mono_data)} mono + {len(sr_data)} stress-relaxation curves")

## 3. Raw-vs-corrected and Native-vs-UV plots

In [ ]:
plotting.plot_raw_vs_corrected(index, mono_data, sr_data, cfg.DIR_RAW_VS_CORR)
plotting.plot_comparison_native_uv(index, pairs, mono_data, sr_data, cfg.DIR_COMPARISON)

# show ALL raw-vs-corrected figures (one per group)
show_all(cfg.DIR_RAW_VS_CORR)

## 4. Mono Hertz fits (loading curve; two contact-point methods)

Both contact-point methods are run below **for comparison** (`"moving"` = moving regression, `"p50"` = data above 50 % of peak). To run only one, comment out its line here and its entry in `save_results` (Step 6).

In [ ]:
mono_loading = {}
for g in index.subgroup_mono:
    for p in index.subgrouped_files[g]:
        df = mono_data.get(p)
        if df is not None and not df.empty:
            mono_loading[p] = get_loading(df)

res_moving = plotting.fit_and_plot_mono(index, pairs, mono_loading, "moving",
                                        "Loading moving regression", cfg.DIR_MONO_FIT,
                                        cfg.PROBE_RADIUS, cfg.POISSON_RATIO)
res_p50 = plotting.fit_and_plot_mono(index, pairs, mono_loading, "p50",
                                     "Loading data above 50 percent peak", cfg.DIR_MONO_FIT,
                                     cfg.PROBE_RADIUS, cfg.POISSON_RATIO)
res_moving.head()

In [ ]:
# show ALL mono Hertz-fit figures (both methods x each Native/UV pair)
show_all(cfg.DIR_MONO_FIT)

## 5. Stress-relaxation 1-term Prony fits

In [ ]:
prony_df = plotting.fit_and_plot_sr(index, pairs, sr_data, cfg.DIR_COMPARISON, cfg.SR_FIT_MAX_TIME)
prony_df

In [ ]:
# show ALL Native-vs-UV comparison overlays + stress-relaxation Prony fits
show_all(cfg.DIR_COMPARISON)

## 6. Save all fit parameters back into the metadata sheet

In [ ]:
final = save_results(cfg.SAMPLE_SHEET, {"moving": res_moving, "p50": res_p50}, prony_df)
final